# 03 – Merge Pre-1997 Legacy Decisions into EC Antitrust Master

## Ziel

Dieses Notebook integriert die Legacy Formal Decisions (pre-1997) in den bestehenden EC-Antitrust-Master.
Das Ergebnis ist eine aktualisierte `ec_antitrust_master.csv` mit genau diesen Spalten:
`ec_case_number`, `case_title`, `date`, `date_source`, `type`, `celex_no`, `document_type`, `document_url`


## 1. Imports


In [1]:
import pandas as pd
from pathlib import Path

print("Imports OK")


Imports OK


## 2. Pfade


In [2]:
MASTER_PATH = Path("data/processed/ec_antitrust_master.csv")
LEGACY_PATH = Path("data/processed/ec_legacy_formal_decisions.csv")
OUTPUT_PATH = Path("data/processed/ec_antitrust_master.csv")

FINAL_COLUMNS = ["ec_case_number", "case_title", "date", "date_source", "type", "celex_no", "document_type", "document_url"]

print(f"Master : {MASTER_PATH}")
print(f"Legacy : {LEGACY_PATH}")
print(f"Output : {OUTPUT_PATH}")


Master : data\processed\ec_antitrust_master.csv
Legacy : data\processed\ec_legacy_formal_decisions.csv
Output : data\processed\ec_antitrust_master.csv


## 3. Master laden


In [3]:
df_master = pd.read_csv(MASTER_PATH, dtype=str)
print(f"Master geladen: {len(df_master)} Zeilen, Spalten: {df_master.columns.tolist()}")
df_master.head(3)


Master geladen: 748 Zeilen, Spalten: ['ec_case_number', 'case_title', 'date', 'date_source', 'type', 'document_type', 'document_url']


,ec_case_number,case_title,date,date_source,type,document_type,document_url
0,AT.35803,IPEX Consortium,1996-01-29,attachment_document,AtStandardATCCase,case,https://ec.europa.eu/competition/antitrust/cas...
1,AT.34950,ECO EMBALLAGES,2001-06-15,case_last_decision,AtStandardATCCase,none,NaN
2,AT.39172,Electricity sector inquiry,2007-01-10,case_last_decision,AtSectorInquiryCase,none,NaN


## 4. Legacy laden


In [4]:
df_legacy = pd.read_csv(LEGACY_PATH, dtype=str)
print(f"Legacy geladen: {len(df_legacy)} Zeilen, Spalten: {df_legacy.columns.tolist()}")
df_legacy.head(3)


Legacy geladen: 437 Zeilen, Spalten: ['year', 'source_url', 'title', 'date_raw', 'decision_type', 'official_journal', 'case_numbers', 'celex_no', 'document_url', 'date', 'date_source']


,year,source_url,title,date_raw,decision_type,official_journal,case_numbers,celex_no,document_url,date,date_source
0,1964,https://ec.europa.eu/competition/antitrust/clo...,Deca,22.10.1964,Negative clearance Art.81(1) [ex 85(1)],L - 31/10/1964 Page : 2761,IV/71,31964D0599,https://publications.europa.eu/resource/celex/...,1964-10-22,legacy_raw
1,1964,https://ec.europa.eu/competition/antitrust/clo...,Grundig-Consten,23.09.1964,Infringement Art.81 [ex 85],L - 20/10/1964 Page : 2545,IV/3344; IV/4,31964D0566,https://publications.europa.eu/resource/celex/...,1964-09-23,legacy_raw
2,1964,https://ec.europa.eu/competition/antitrust/clo...,Nicholas Freres + Vitapro,30.07.1964,Negative clearance Art.81(1) [ex 85(1)],L - 26/08/1964 Page : 2287,IV/95,31964D0502,https://publications.europa.eu/resource/celex/...,1964-07-30,legacy_raw


## 5. Spalten auf Zielschema mappen


In [5]:
# Master: Spalten bereits passend (ec_case_number, case_title, date, date_source, type)
# celex_no fehlt im Master -> leer lassen
df_master_mapped = df_master.reindex(columns=FINAL_COLUMNS)

print(f"Master gemappt: {df_master_mapped.shape}")
df_master_mapped.head(3)


Master gemappt: (748, 8)


,ec_case_number,case_title,date,date_source,type,celex_no,document_type,document_url
0,AT.35803,IPEX Consortium,1996-01-29,attachment_document,AtStandardATCCase,NaN,case,https://ec.europa.eu/competition/antitrust/cas...
1,AT.34950,ECO EMBALLAGES,2001-06-15,case_last_decision,AtStandardATCCase,NaN,none,NaN
2,AT.39172,Electricity sector inquiry,2007-01-10,case_last_decision,AtSectorInquiryCase,NaN,none,NaN


In [6]:
# Legacy: Spalten umbenennen und auf Zielschema bringen
# Vorhandene Spalten in ec_legacy_formal_decisions.csv:
#   year, title, date_raw, date, date_source, decision_type, official_journal,
#   celex_no, case_numbers, source_url, document_url
df_legacy_mapped = df_legacy.rename(columns={
    "case_numbers": "ec_case_number",
    "title":        "case_title",
    "date":         "date",
    "date_source":  "date_source",
    "celex_no":     "celex_no",
})

# type als konstanter Wert
df_legacy_mapped["type"] = "formal_decision"

df_legacy_mapped = df_legacy_mapped.reindex(columns=FINAL_COLUMNS)

print(f"Legacy gemappt: {df_legacy_mapped.shape}")
df_legacy_mapped.head(3)


Legacy gemappt: (437, 8)


,ec_case_number,case_title,date,date_source,type,celex_no,document_type,document_url
0,IV/71,Deca,1964-10-22,legacy_raw,formal_decision,31964D0599,NaN,https://publications.europa.eu/resource/celex/...
1,IV/3344; IV/4,Grundig-Consten,1964-09-23,legacy_raw,formal_decision,31964D0566,NaN,https://publications.europa.eu/resource/celex/...
2,IV/95,Nicholas Freres + Vitapro,1964-07-30,legacy_raw,formal_decision,31964D0502,NaN,https://publications.europa.eu/resource/celex/...


## 6. Zusammenführen


In [7]:
df_combined = pd.concat([df_master_mapped, df_legacy_mapped], ignore_index=True)
print(f"Kombiniert: {len(df_combined)} Zeilen")


Kombiniert: 1185 Zeilen


## 7. Finale Spaltenreihenfolge


In [8]:
df_final = df_combined[FINAL_COLUMNS]
print(f"Finale Spalten: {df_final.columns.tolist()}")
print(f"Finale Shape  : {df_final.shape}")
df_final.head(5)


Finale Spalten: ['ec_case_number', 'case_title', 'date', 'date_source', 'type', 'celex_no', 'document_type', 'document_url']
Finale Shape  : (1185, 8)


,ec_case_number,case_title,date,date_source,type,celex_no,document_type,document_url
0,AT.35803,IPEX Consortium,1996-01-29,attachment_document,AtStandardATCCase,NaN,case,https://ec.europa.eu/competition/antitrust/cas...
1,AT.34950,ECO EMBALLAGES,2001-06-15,case_last_decision,AtStandardATCCase,NaN,none,NaN
2,AT.39172,Electricity sector inquiry,2007-01-10,case_last_decision,AtSectorInquiryCase,NaN,none,NaN
3,AT.39294,Microsoft (ECIS complaint),2010-06-25,attachment_document,AtStandardATCCase,NaN,case,https://ec.europa.eu/competition/antitrust/cas...
4,AT.39173,Gas sector inquiry,2007-01-10,case_last_decision,AtSectorInquiryCase,NaN,none,NaN


## 8. Qualitätszusammenfassung


In [9]:
total        = len(df_final)
filled       = df_final["date"].notna().sum()
missing      = df_final["date"].isna().sum()
by_source    = df_final["date_source"].value_counts(dropna=False)

# Jahresbereich (nur gültige Daten)
years = pd.to_datetime(df_final["date"], errors="coerce").dt.year.dropna()
first_year = int(years.min()) if len(years) > 0 else None
last_year  = int(years.max()) if len(years) > 0 else None

print("=" * 50)
print("EC CASES – DATE QUALITY SUMMARY")
print("=" * 50)
print(f"  Total EC cases          : {total}")
print(f"  Cases with date filled  : {filled}  ({filled/total*100:.1f}%)")
print(f"  Cases still missing     : {missing}  ({missing/total*100:.1f}%)")
print(f"  First year              : {first_year}")
print(f"  Last year               : {last_year}")
print()
print("  Counts by date_source:")
for src, cnt in by_source.items():
    print(f"    {str(src):<25} {cnt}")
print("=" * 50)


EC CASES – DATE QUALITY SUMMARY
  Total EC cases          : 1185
  Cases with date filled  : 1185  (100.0%)
  Cases still missing     : 0  (0.0%)
  First year              : 1964
  Last year               : 2025

  Counts by date_source:
    case_last_decision        515
    legacy_raw                437
    initiation                87
    case_oj                   85
    attachment_document       61


## 9. ec_antitrust_master.csv speichern


In [10]:
df_final.to_csv(OUTPUT_PATH, index=False, encoding="utf-8")
print(f"✅ Gespeichert: {OUTPUT_PATH.resolve()}")
print(f"   {len(df_final)} Zeilen, {len(df_final.columns)} Spalten")


✅ Gespeichert: C:\vsCodeWorkspace\masterZHAW\masterarbeit\data_analysis_master\data\processed\ec_antitrust_master.csv
   1185 Zeilen, 8 Spalten
